In [1]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tensorflow import keras
from keras import backend as K
from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()


os.environ["CUDA_VISIBLE_DEVICES"]="0"

## Useful functions 

In [2]:
def filter_label(df, y, path, label):
    #divide dataset based on the label[]
     
    output_df = df[np.where(y == label)[0]]
    
    res_df =  df[np.where(y != label)[0]]
    
    output_y = y[np.where(y == label)[0]]
    res_y = y[np.where(y != label)[0]]
    
    output_path = path[np.where(y == label)[0]]
    
    res_path =  path[np.where(y != label)[0]]
    

    
    return  output_df, output_y, output_path, res_df, res_y, res_path

 
# Function to obtain labels from a DataFrame and a CSV file
def obtain_labels(df, label_path):
    
    labels = pd.read_csv(label_path, header=0)
    
    y = []
    x = []
    
    for index, row in df.iterrows():
        #print(row["asm_id"].split(".")[0])
        hash_id = row["asm_id"].split(".")[0]
        if hash_id in labels['asm_id'].values: 
            row = row.drop("asm_id")
            x.append(row)
            y.append(labels[labels["asm_id"] == hash_id]["Class"])
            

    
    return np.array(x), np.array(y)

# Function to change the attack label to a binary format
def change_attack_label(x):
    label = [1.]
    return label



def load_image_malware(image_path, label_path):
    
    labels = pd.read_csv(label_path, header=0)
    
    x = []
    y = []
    paths = []
            
    
    
    for filename in os.listdir(image_path):
        if filename.endswith(".png"):
            hash_id = filename.split(".")[0]
            if hash_id in labels['asm_id'].values: 
                f = os.path.join(image_path, filename)
                image = Image.open(f).convert('RGB')
                image = image.resize((56, 56), Image.ANTIALIAS)
                image = np.array(image, dtype=int)
                x.append(image)
                y.append(labels[labels["asm_id"] == hash_id]["Class"])
                paths.append(image_path + "/" + filename)

         
            
    x = np.asarray(x)
    y = np.asarray(y)
                       
    x = x.astype('float32') / 255.
    paths = np.array(paths, dtype=object)
    
    
        
    return x, y, paths





def load_image_normal(directory_path):
    image_list = []
    image_size_limit = 178956970  # Maximum allowed pixels per image
    
    paths = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".jpg") or filename.endswith(".png") or filename.endswith(".jpeg"):
            file_path = os.path.join(directory_path, filename)
            try:
                Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError check for large images
                with Image.open(file_path) as img:
                    Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError check for large images
                    # Check if the image size is within the allowed limit
                    if img.width * img.height <= image_size_limit:
                        img = img.convert('RGB')
                        img = img.resize((56, 56), Image.ANTIALIAS)
                        img_array = np.array(img, dtype=int)
                        image_list.append(img_array)
                        paths.append(directory_path + "/" + filename)
                    else:
                        print(f"Image {filename} exceeds the size limit of {image_size_limit} pixels and will be skipped.")
            except (Image.DecompressionBombError, OSError) as e:
                print(f"Error loading image {filename}: {e}")

    image_list = np.asarray(image_list)
    image_list = image_list.astype('float32') / 255.
    paths = np.array(paths, dtype=object)

    return image_list, paths



# Custom metrics for model evaluation
def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))



## Load malware data

In [ ]:
label_path = "../../data/labels/new_gmm_labels_id.csv"
img_path ="../../data/image_features/Big15_2"
malware_x, malware_y, malware_path =load_image_malware(img_path, label_path)

## Load normal data source

In [ ]:
 # Load normal data source
img_path ="../../data/image_features/benign_source/dataset1"
source_normal_x_1, source_normal_path_1=load_image_normal(img_path)
source_normal_y_1 = np.zeros((source_normal_x_1.shape[0],1))

img_path ="../../data/image_features/benign_source/dataset2"
source_normal_x_2, source_normal_path_2=load_image_normal(img_path)
source_normal_y_2 = np.zeros((source_normal_x_2.shape[0],1))

img_path ="../../data/image_features/benign_source/dataset3"
source_normal_x_3, source_normal_path_3=load_image_normal(img_path)
source_normal_y_3 = np.zeros((source_normal_x_3.shape[0],1))

img_path ="../../data/image_features/benign_source/dataset4"
source_normal_x_4, source_normal_path_4 =load_image_normal(img_path)
source_normal_y_4 = np.zeros((source_normal_x_4.shape[0],1))


source_normal_x = np.concatenate((source_normal_x_1, source_normal_x_2, source_normal_x_3, source_normal_x_4), axis = 0)
source_normal_y = np.concatenate((source_normal_y_1, source_normal_y_2, source_normal_y_3, source_normal_y_4), axis = 0)
source_normal_path = np.concatenate((source_normal_path_1, source_normal_path_2, source_normal_path_3, source_normal_path_4), axis = 0)

## Load normal data target

In [ ]:
# Load normal data target
img_path ="../../data/image_features/benign_target/dataset1"
target_normal_x_1, target_normal_path_1 =load_image_normal(img_path)
target_normal_y_1 = np.zeros((target_normal_x_1.shape[0],1))

img_path ="../../data/image_features/benign_target/dataset2"
target_normal_x_2, target_normal_path_2 =load_image_normal(img_path)
target_normal_y_2 = np.zeros((target_normal_x_2.shape[0],1))

img_path ="../../data/image_features/benign_target/dataset3"
target_normal_x_3, target_normal_path_3 =load_image_normal(img_path)
target_normal_y_3 = np.zeros((target_normal_x_3.shape[0],1))

img_path ="../../data/image_features/benign_target/dataset4"
target_normal_x_4, target_normal_path_4 =load_image_normal(img_path)
target_normal_y_4 = np.zeros((target_normal_x_4.shape[0],1))

#merge
target_normal_x = np.concatenate((target_normal_x_1, target_normal_x_2, target_normal_x_3, target_normal_x_4), axis = 0)
target_normal_y = np.concatenate((target_normal_y_1, target_normal_y_2, target_normal_y_3, target_normal_y_4), axis = 0)
target_normal_path = np.concatenate((target_normal_path_1, target_normal_path_2, target_normal_path_3, target_normal_path_4), axis = 0)

## C1

In [6]:

target_malware_x, target_malware_y, target_malware_path,  source_malware_x, source_malware_y, source_malware_path=  filter_label(malware_x, malware_y, malware_path, [1])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Target; {}".format(target_malware_path.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))
print("Source {}".format(source_malware_path.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Target: {}".format(target_normal_path.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))
print("Source {}".format(source_normal_path.shape))


source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)


source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)
source_path =  np.concatenate((source_malware_path, source_normal_path), axis = 0)


target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)
target_path = np.concatenate((target_malware_path, target_normal_path), axis = 0)


source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)



print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))



source_x_train, source_x_test,\
source_y_train, source_y_test,\
source_path_train, source_path_test = train_test_split(source_x, source_y, source_path, test_size=0.25, random_state=42)

target_x_train, target_x_test,\
target_y_train, target_y_test,\
target_path_train, target_path_test = train_test_split(target_x, target_y, target_path, test_size=0.5, random_state=42)
        
    
print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))

      

Malware data ...
Target: (3202, 56, 56, 3)
Target; (3202, 1)
Target; (3202,)
Source (7502, 56, 56, 3)
Source (7502, 1)
Source (7502,)
Normal data ...
Target: (9150, 56, 56, 3)
Target: (9150, 1)
Target: (9150,)
Source (8208, 56, 56, 3)
Source (8208, 1)
Source (8208,)
Combined data ...
Target: (12352, 56, 56, 3)
Target: (12352, 2)
Source (15710, 56, 56, 3)
Source (15710, 2)
train test data ...
Target train: (6176, 56, 56, 3)
Target train: (6176, 2)
Target test: (6176, 56, 56, 3)
Target test: (6176, 2)
Source train: (11782, 56, 56, 3)
Source train: (11782, 2)
Source test: (3928, 56, 56, 3)
Source test: (3928, 2)


### Load MaxDIRep

In [ ]:

generator = keras.models.load_model("../../data/stepI_trained_models/big15/c1/generator")
classifier = keras.models.load_model("../../data/stepI_trained_models/big15/c1/classifier")


y_target_class_pred = classifier.predict(generator(target_x_train)).argmax(1)

# 1. Original noisy‐label accuracy on training data
y_noisy = y_target_class_pred  
y_true_all = target_y_train.argmax(axis=1)
acc_orig = accuracy_score(y_true_all, y_target_class_pred)


In [9]:

Z_maps = generator.predict(target_x_train)        # shape = (N, 12, 12, 64)
N = Z_maps.shape[0]

Z = Z_maps.reshape(N, -1)    
 


193/193 [==============================] - 0s 1ms/step


### local outlier factor

In [10]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)           # best-class probability

# 3. Build masks: LOF-only, confidence-only, and combined
keep_lof   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
lof_contam = 0.2 # fraction of outliers per class
conf_thresh = 0.95  # confidence cutoff

#y_noisy = y_target_class_pred
# 4. Per-class LOF filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # fit LOF on class-c embeddings
    lof = LocalOutlierFactor(n_neighbors=50, contamination=lof_contam)
    preds = lof.fit_predict(Zc)   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update LOF-only mask
    keep_lof[idx[inliers]] = True

    # update confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
# y_true_all  = target_y_train.argmax(axis=1)

y_true_lof   = y_true_all[keep_lof]
y_pred_lof   = y_noisy[keep_lof]


y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]


y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig  = accuracy_score(y_true_all, y_noisy)
acc_lof   = accuracy_score(y_true_lof, y_pred_lof)
acc_conf  = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"After LOF-only                : {acc_lof:.2%} "
      f"({keep_lof.sum()}/{N} ≈ {keep_lof.mean():.1%} retained)")
print(f"After confidence-only         : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After LOF + confidence        : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")

    

193/193 [==============================] - 0s 1ms/step
Original accuracy             : 91.52% on 6176 samples
After LOF-only                : 93.44% (4940/6176 ≈ 80.0% retained)
After confidence-only         : 96.53% (5309/6176 ≈ 86.0% retained)
After LOF + confidence        : 97.23% (4372/6176 ≈ 70.8% retained)



In [11]:
target_x_train_filtered = target_x_train[keep_combo]
target_pred_train_filtered = y_pred_combo
target_path_train_filtered  = target_path_train[keep_combo]

### GMM

In [13]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: GMM-only, confidence-only, and combined
keep_gmm   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
gmm_contam = 0.20    # fraction of outliers per class
conf_thresh = 0.95   # confidence cutoff

# y_noisy = y_target_class_pred

# 4. Per-class GMM filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit a single-component GMM
    gmm = GaussianMixture(n_components=1,
                          covariance_type='full',
                          reg_covar=1e-6,
                          random_state=0)
    gmm.fit(Zc)

    # compute log-likelihoods
    log_probs = gmm.score_samples(Zc)           # shape = (n_c,)

    # GMM-only mask (inliers above quantile)
    thresh = np.percentile(log_probs, gmm_contam * 100)
    inliers = log_probs > thresh
    keep_gmm[idx[inliers]] = True

    # confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all   = target_y_train.argmax(axis=1)
y_true_gmm   = y_true_all[keep_gmm]
y_pred_gmm   = y_noisy[keep_gmm]

y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_gmm  = accuracy_score(y_true_gmm, y_pred_gmm)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo= accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy               : {acc_orig:.2%} on {N} samples")
print(f"After GMM-only                  : {acc_gmm:.2%} "
      f"({keep_gmm.sum()}/{N} ≈ {keep_gmm.mean():.1%} retained)")
print(f"After confidence-only           : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After GMM + confidence          : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")




193/193 [==============================] - 0s 1ms/step
Original accuracy               : 91.52% on 6176 samples
After GMM-only                  : 91.09% (4940/6176 ≈ 80.0% retained)
After confidence-only           : 96.53% (5309/6176 ≈ 86.0% retained)
After GMM + confidence          : 96.89% (4145/6176 ≈ 67.1% retained)



### One class svm 

In [14]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: One-Class SVM only, confidence-only, and combined
keep_ocsvm = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
svm_nu     = 0.2      # fraction of outliers per class
aic_conf_thresh = 0.95   # confidence cutoff

# 4. Per-class One-Class SVM filtering + confidence gating
y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit One-Class SVM
    ocsvm = OneClassSVM(nu=svm_nu, kernel='rbf', gamma='auto')
    ocsvm.fit(Zc)
    preds = ocsvm.predict(Zc)                   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update One-Class SVM mask
    keep_ocsvm[idx[inliers]] = True

    # update confidence mask
    conf_mask = conf[idx] >= aic_conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all     = target_y_train.argmax(axis=1)
y_true_ocsvm   = y_true_all[keep_ocsvm]
y_pred_ocsvm   = y_noisy[keep_ocsvm]

y_true_conf    = y_true_all[keep_conf]
y_pred_conf    = y_noisy[keep_conf]

y_true_combo   = y_true_all[keep_combo]
y_pred_combo   = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig   = accuracy_score(y_true_all, y_noisy)
acc_ocsvm  = accuracy_score(y_true_ocsvm, y_pred_ocsvm)
acc_conf   = accuracy_score(y_true_conf, y_pred_conf)
acc_combo  = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy                : {acc_orig:.2%} on {N} samples")
print(f"After One-Class SVM only         : {acc_ocsvm:.2%} "
      f"({keep_ocsvm.sum()}/{N} ≈ {keep_ocsvm.mean():.1%} retained)")
print(f"After confidence-only            : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After One-Class SVM + confidence : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")



193/193 [==============================] - 0s 1ms/step
Original accuracy                : 91.52% on 6176 samples
After One-Class SVM only         : 90.77% (4929/6176 ≈ 79.8% retained)
After confidence-only            : 96.53% (5309/6176 ≈ 86.0% retained)
After One-Class SVM + confidence : 96.61% (4126/6176 ≈ 66.8% retained)



### Mahalanobis-distance

In [15]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
from scipy.stats import chi2
from sklearn.metrics import accuracy_score


# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)    # best-class probability

# 3. Build masks: Mahalanobis-only, confidence-only, and Mahalanobis+confidence
keep_maha = np.zeros(N, dtype=bool)
keep_conf = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
maha_alpha = 0.8    # χ² percentile threshold
conf_thresh = 0.95  # confidence cutoff

# 4. Per-class filtering
# y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # Mahalanobis distances
    cov_est = EmpiricalCovariance().fit(Zc)
    m2 = cov_est.mahalanobis(Zc)
    maha_thresh = chi2.ppf(maha_alpha, df=Zc.shape[1])
    maha_mask = m2 < maha_thresh

    # Confidence mask
    conf_mask = conf[idx] >= conf_thresh

    # Update masks
    keep_maha[idx[maha_mask]] = True
    keep_conf[idx[conf_mask]] = True
    keep_combo[idx[maha_mask & conf_mask]] = True

# 5. Slice subsets
y_true_all = target_y_train.argmax(axis=1)
y_true_maha = y_true_all[keep_maha]
y_pred_maha = y_noisy[keep_maha]

y_true_conf = y_true_all[keep_conf]
y_pred_conf = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_maha = accuracy_score(y_true_maha, y_pred_maha)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy                : {acc_orig:.2%} on {N} samples")
print(f"After Mahalanobis-only           : {acc_maha:.2%} "
      f"({keep_maha.sum()}/{N} ≈ {keep_maha.mean():.1%} retained)")
print(f"After confidence-only            : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After Mahalanobis + confidence   : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")



193/193 [==============================] - 0s 1ms/step
Original accuracy                : 91.52% on 6176 samples
After Mahalanobis-only           : 90.44% (4287/6176 ≈ 69.4% retained)
After confidence-only            : 96.53% (5309/6176 ≈ 86.0% retained)
After Mahalanobis + confidence   : 96.83% (3535/6176 ≈ 57.2% retained)


### Isolation forest

In [16]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble     import IsolationForest
from sklearn.metrics      import accuracy_score

# 2. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)


# 4. Softmax probabilities + confidence
probs = classifier.predict(Z) # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 5. Build two masks:
#    - IsolationForest only
#    - IsolationForest + confidence
keep_iforest = np.zeros(N, dtype=bool)
keep_conf   = np.zeros(N, dtype=bool)
keep_combo   = np.zeros(N, dtype=bool)

iso_contam   = 0.2  # fraction of outliers per class
conf_thresh  = 0.95  # confidence cutoff

for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit IsolationForest on class-c embeddings
    iso = IsolationForest(contamination=iso_contam, random_state=0)
    iso.fit(Zc)
    preds = iso.predict(Zc)                     # +1 inlier, -1 outlier
    inliers = (preds == 1)

    # mark kept for IF only
    keep_iforest[idx[inliers]] = True

    # combine with high-confidence
    conf_mask = conf[idx] > conf_thresh
    keep_conf[idx[conf_mask]] = True
    
    keep_combo[idx[inliers & conf_mask]] = True

# 6. Slice out subsets
y_true_iforest  = y_true_all[keep_iforest]
y_pred_iforest  = y_noisy[keep_iforest]


y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo    = y_true_all[keep_combo]
y_pred_combo    = y_noisy[keep_combo]

# 7. Compute & print accuracies
acc_orig    = accuracy_score(y_true_all, y_noisy)
acc_iforest = accuracy_score(y_true_iforest, y_pred_iforest)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo   = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"After IsolationForest only    : {acc_iforest:.2%} "
      f"({keep_iforest.sum()}/{N} ≈ {keep_iforest.mean():.1%} retained)")
print(f"After confidence filter     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After + confidence filter     : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")



193/193 [==============================] - 0s 1ms/step
Original accuracy             : 91.52% on 6176 samples
After IsolationForest only    : 90.71% (4940/6176 ≈ 80.0% retained)
After confidence filter     : 96.53% (5309/6176 ≈ 86.0% retained)
After + confidence filter     : 96.57% (4136/6176 ≈ 67.0% retained)


### Save the source and target 

In [12]:


np.savez_compressed('../../data/stepII_constructed_datasets/big15/C1/source_train.npz',
                    source_path_train=source_path_train, source_y_train = source_y_train.argmax(axis=1))
np.savez_compressed('../../data/stepII_constructed_datasets/big15/C1/source_test.npz',
                    source_path_test=source_path_test, source_y_test = source_y_test.argmax(axis=1))
np.savez_compressed('../../data/stepII_constructed_datasets/big15/C1/target_train_filtered.npz',
                    target_path_train_filtered=target_path_train_filtered, target_pred_train_filtered=target_pred_train_filtered,target_true_train_filtered=y_true_combo)
np.savez_compressed('../../data/stepII_constructed_datasets/big15/C1/target_train.npz',
                    target_path_train=target_path_train,target_y_train=target_y_train.argmax(axis=1))
np.savez_compressed('../../data/stepII_constructed_datasets/big15/C1/target_test.npz',
                    target_path_test=target_path_test,target_y_test=target_y_test.argmax(axis=1))


